#  Mini-Project 2: Travelling between Inner and Outer London, Peak or Off-Peak Hours

**DS105W Mini-Project 2, Data for Data Science (Winter Term 2025/2026)**

<div style="font-family: system-ui; padding: 20px 30px 20px 20px; background-color: #FFFFFF; border-left: 8px solid #ED9255; border-radius: 8px; box-shadow: 0 4px 12px rgba(0, 0, 0, 0.1);max-width:600px;color:#212121;">

**Student Notebook**
- 📅 Date: 1 Apr 2026 (due date)
- 👤 Name: Mingyun Wang
- 📛 Candidate Number: 70643
- 🎯 Purpose of this Notebook: Transforming Data


</div>

## The Overall Plan is here 👇:
You probably have been able to access this overall plan via README.md.


- the Inner London area is **LSE Main Campus** (as a default set by the mini project requirement, and controlling this as a single destination helps derive simplicity and standardisation over a range of omitted variables), and the Outer London boroughs: **Ealing, Sutton, Barking, and Enfield**. Reasons why I chose these are briefed in NB01.
- Some **sample postcodes** will be selected for API requests regarding journey plans based on their LSOA directories and usertypes. (*See NB01 for how those postcodes are chosen*)

- there are a few types of the investigated variables: *(note this will also appear in NB02)*
  1. `Identity Information:`
    - **postcode** (of specific spots of LSOAs subject to boroughs)
    - **timeband** (peak or offpeak)
    - **borough** (one of Ealing, Sutton, Barking, or Enfield)
    - **usertype** (binary, 0 for residential, 1 for business)
    - **IMD** (index, how deprive an LSOA is)
  2. `Time-Related Information:`
    - **duration** (averaged for inward and outword journeys)
    - **walking minutes** (averaged)
    - **transport minutes** (averaged, non-walking time during a journey, or AKA time spent on transportations)
  3. `Mode-Related Information:`
    - **complexity** (averaged, the number of transfers/transits/switches of travelling mode, obtainable by counting the number of a journey's legs)
    - **major mode** (the non-walking rode that one is estimated to spend the most time on during a journey)
  4. `Cost Information:`
    - **fare total cost** (averaged)
  - `NOTE` that several variables, as marked, are the **average** of a sample journey from outer London to inner London and the average of a sample journey form inner London to outer London. This is to balance the subtle difference between travelling inwards and outwards due to a range of reasons, like the convenience of getting to bus stops and underground stations.

- **Peak and Off-Peak Hours**: (*for more detailed definition and reason, see NB01*)
  - for **Peak Hours**, data are collected for **either 2026 Mar 19 (Thu) 08:00 or 2026 Mar 19 (Thu) 17:00**
  - for **Off-Peak Hours**, data are collected for **only 2026 Mar 19 (Thu) 12:00**

- There is a **Trial** Workflow and an **Extension** Workflow for NB01 and NB02:
  - This arrangement is because I would like to write my Data Collection and Transformation workflows with Ealing into functions, so that the amount of work will be reduced for more boroughs in terms of these repetitive jobs.
  - The **Trial** workflow picks one outer london area and one inner london area only, specifically the inner London spot will be **LSE Main Campus** (as the project's default) and the outer London borough will be **Ealing** (not a single spot, including the districts of Acton, Ealing, Greenford, Hanwell, Northolt, Perivale and Southall)
  - These choices pretty make sense because both places are Bustling Busy Business areas surrounded by/involving a wide range of businesses and residential spots. Then it would be quite reasonable to expect that we could see more intense commuting flows from A to B and a clearer difference between peak/off-peak hours.
  - The **Extension** workflow is expected to engage the rest of the boroughs: Sutton, Barking, and Enfield.
  - I complete a whole cycle of the Trial, and only then will I work with the Extension
- NB03 pools data together for analyses

## 2.0, In This Notebook 📓:


1. **My aim is to make a long, plottable table for each of the four boroughs I've selected in NB01. Each borough's table should contain these types of columns:**
  - **Identity Information**:
    - postcode (of specific spots of LSOAs subject to boroughs)
    - timeband (peak or offpeak)
    - borough (one of Ealing, Sutton, Barking, or Enfield)
    - usertype (binary, 0 for residential, 1 for business)
    - IMD (index, how deprive an LSOA is)
  - **Time-Related Information**:
    - duration (averaged for inward and outword journeys)
    - walking minutes (averaged)
    - transport minutes (averaged, non-walking time during a journey, or AKA time spent on transportations)
  - **Mode-Related Information**:
    - complexity (averaged, the number of transfers/transits/switches of travelling mode)
    - major mode (the non-walking rode that one is estimated to spend the most time on during a journey)
  - **Cost Information**:
    - fare total cost (averaged)
2. I'll work with the Trial workflow first, meaning: I'll be sorting things out for the borough Ealing manually before applying the same workflow to the rest of the boroughs. This is essential because accessing each variable I want from raw data helps me know the locations of each piece of data. It will be only then that I'll be able to summarise these ways of accessing data into concise and convenient fuctions.

3. I'll write and upgrade my procedure that has worked for Ealing into some python functions, so that I can extend the workflow by applying the functions directly to the rest of the boroughs.

4. Processed data for all boroughs will be saved as .csv files in the directory '/data/processed'.

## 2.1, TRIAL: Ealing
P.S., this time, instead of directly putting the functions I wrote, I'd first demonstrate how one would manually do the workflow of the functions and extract necessary variables from the nested structure. I believe this will help clarify the logic of my functions and explain specific variables I define. Though this time's function seems lengthier than those in NB01 or mini-project 1, it could be gradually built up logically step-by-step following the natural logic one would try to access the nested, raw data structure for a borough during a specific time band.

`Again, this section is not necessary if you simply would like to see the function that has led to the resulting dataframes. It is only necessary if you care about how the function was constructed step-by-step. So feel free to skip this section if you'd like, since this section would seem kind of long.`

### 2.1.0, Libraries

In [18]:
import csv
import json
import pandas as pd
import numpy as np
import datetime

### 2.1.1, How to Extract Information for Each Journey?
`Jump if you aren't interested in this.` I'll still briefly introduce how the function works in the next section. Here, I'll use the first response journey (during peak hours, between a postcode in Ealing and LSE) as my example. Each step of decomposing the nested structure will become a core step in the upcoming function (in section 2.1.2).

Load Ealing_peak and browse:

In [19]:
ealing_peak = pd.read_json('../data/raw/ealing_peak_data.json')
print('First 3 rows of ealing_peak:')
ealing_peak.head(3)

First 3 rows of ealing_peak:


,$type,journeys,lines,stopMessages,recommendedMaxAgeMinutes,searchCriteria,journeyVector
0,Tfl.Api.Presentation.Entities.JourneyPlanner.I...,[{'$type': 'Tfl.Api.Presentation.Entities.Jour...,[{'$type': 'Tfl.Api.Presentation.Entities.Line...,[],5,{'$type': 'Tfl.Api.Presentation.Entities.Journ...,{'$type': 'Tfl.Api.Presentation.Entities.Journ...
1,Tfl.Api.Presentation.Entities.JourneyPlanner.I...,[{'$type': 'Tfl.Api.Presentation.Entities.Jour...,[{'$type': 'Tfl.Api.Presentation.Entities.Line...,[],5,{'$type': 'Tfl.Api.Presentation.Entities.Journ...,{'$type': 'Tfl.Api.Presentation.Entities.Journ...
2,Tfl.Api.Presentation.Entities.JourneyPlanner.I...,[{'$type': 'Tfl.Api.Presentation.Entities.Jour...,[{'$type': 'Tfl.Api.Presentation.Entities.Line...,[],5,{'$type': 'Tfl.Api.Presentation.Entities.Journ...,{'$type': 'Tfl.Api.Presentation.Entities.Journ...


dive into one single journey (the very first journey, call it journey_0):

In [20]:
pd.json_normalize(ealing_peak['journeys'][0], sep='_')

,$type,startDateTime,duration,arrivalDateTime,alternativeRoute,legs,fare_$type,fare_totalCost,fare_fares,fare_caveats
0,Tfl.Api.Presentation.Entities.JourneyPlanner.J...,2026-10-22T08:04:00,60,2026-10-22T09:04:00,False,[{'$type': 'Tfl.Api.Presentation.Entities.Jour...,Tfl.Api.Presentation.Entities.JourneyPlanner.J...,565,[{'$type': 'Tfl.Api.Presentation.Entities.Jour...,[{'$type': 'Tfl.Api.Presentation.Entities.Jour...
1,Tfl.Api.Presentation.Entities.JourneyPlanner.J...,2026-10-22T08:08:00,65,2026-10-22T09:13:00,False,[{'$type': 'Tfl.Api.Presentation.Entities.Jour...,Tfl.Api.Presentation.Entities.JourneyPlanner.J...,480,[{'$type': 'Tfl.Api.Presentation.Entities.Jour...,[{'$type': 'Tfl.Api.Presentation.Entities.Jour...
2,Tfl.Api.Presentation.Entities.JourneyPlanner.J...,2026-10-22T08:18:00,66,2026-10-22T09:24:00,False,[{'$type': 'Tfl.Api.Presentation.Entities.Jour...,Tfl.Api.Presentation.Entities.JourneyPlanner.J...,480,[{'$type': 'Tfl.Api.Presentation.Entities.Jour...,[{'$type': 'Tfl.Api.Presentation.Entities.Jour...


startDatetime (which helps identify peak/offpeak), duration, legs, and fare_totalCost are all direct keys to the normalised journey_0.

I then take only the shortest journey (in terms of duration) as the only representation of this commute, and this is just to standardise the principle of picking journeys, though the total cost of a shorter journey might tend to be slightly higher.

In [21]:
journey_0 = pd.json_normalize(ealing_peak['journeys'][0], sep='_').dropna()
type(journey_0)
journey_0 = journey_0[journey_0['duration'] == journey_0['duration'].min()].head(1)
journey_0 = journey_0[['startDateTime', 'duration', 'legs', 'fare_totalCost']]
journey_0

,startDateTime,duration,legs,fare_totalCost
0,2026-10-22T08:04:00,60,[{'$type': 'Tfl.Api.Presentation.Entities.Jour...,565


dive into the legs of journey_0:

In [22]:
journey_0_legs = pd.json_normalize(journey_0['legs'].iloc[0], sep='_')
journey_0_legs

,$type,duration,obstacles,departureTime,arrivalTime,routeOptions,disruptions,plannedWorks,distance,isDisrupted,...,mode_type,mode_routeType,mode_status,mode_motType,mode_network,interChangeDuration,interChangePosition,departurePoint_naptanId,departurePoint_stopLetter,departurePoint_individualStopId
0,Tfl.Api.Presentation.Entities.JourneyPlanner.L...,1,[],2026-10-22T08:04:00,2026-10-22T08:05:00,[{'$type': 'Tfl.Api.Presentation.Entities.Jour...,[],[],41.0,False,...,Mode,Unknown,Unknown,100,,NaN,NaN,NaN,NaN,NaN
1,Tfl.Api.Presentation.Entities.JourneyPlanner.L...,14,[{'$type': 'Tfl.Api.Presentation.Entities.Jour...,2026-10-22T08:05:00,2026-10-22T08:19:00,[{'$type': 'Tfl.Api.Presentation.Entities.Jour...,[],[],NaN,False,...,Mode,Unknown,Unknown,3,tfl,5,AFTER,490G00012981,->E,490012981E
2,Tfl.Api.Presentation.Entities.JourneyPlanner.L...,27,[],2026-10-22T08:24:00,2026-10-22T08:51:00,[{'$type': 'Tfl.Api.Presentation.Entities.Jour...,[],[],NaN,False,...,Mode,Unknown,Unknown,1,tfl,NaN,NaN,940GZZLUHGR,NaN,9400ZZLUHGR1
3,Tfl.Api.Presentation.Entities.JourneyPlanner.L...,13,[],2026-10-22T08:51:00,2026-10-22T09:04:00,[{'$type': 'Tfl.Api.Presentation.Entities.Jour...,[],[],588.0,False,...,Mode,Unknown,Unknown,100,,NaN,NaN,940GZZLUHBN,NaN,9400ZZLUHBN1


Here is defined the variables `complexity` and `major_mode`:
  - **complexity** is a generic measure of how complex/frustrating a journey is, and it considers how many legs a journey contains, for each distinct leg also marks a switch/transit in terms of the `mode` of the journey (compared to the previous leg if applicable). This information could be verified by diving into the "mode_id" column of the normalised dataframe of each journey, or viewed in raw data by seeing through the 'journey['legs'][i]['mode']['id']' structure.
  - **major_mode** considers the major/primal mode of non-walking transportation (I don't really expect one would literally travel between outer and inner London on foot). 'Major' means the particular non-walking mode takes the longest time amongst all non-walking legs.

- Besides are defined the variables `walking_mins` and `transport_mins`, recording the **total time** spent on walking and transportations during one distinct journey between outer and inner London.

- **Reflections**🤔: Note that here, walking mins and transport mins of a journey are computed using `.loc[]` and `.sum()`, but they are not in the later function. This is because pandas seems to be particularly serious with boolean operations and keeps raising errors while I used the method here. Then I modified the function using a more direct method with some help from Claude.

- **Reflections**🤔: I found `.groupby()` pretty helpful here computing the duration on each mode of non-walking transportation/mode. Each mode's duration is just the durations within 'legs' summed with respect to (grouped by) distinctive modes. Then `.idmax()` directly identifies the label (mode id/name) featuring the longest duration sum.

In [23]:
walking_0 = journey_0_legs['mode_id'] == 'walking'    # tag walking legs True and display as a Boolean serie
print('walking legs will be tagged True')
display(walking_0)
non_walking_0 = journey_0_legs['mode_id'] != 'walking'    # and by contrast, those tagged False are non-walking legs

journey_0['walking_mins'] = journey_0_legs.loc[walking_0, 'duration'].sum()    # the sum of durations of walking legs: this is how long you'll be walking for this journey
journey_0['transport_mins'] = journey_0_legs.loc[~walking_0, 'duration'].sum()    # the sum of durations of non-walking legs: total time you'll be taking transportations/'travel modes'
mode_mins_0 = journey_0_legs[~walking_0].groupby('mode_id')['duration'].sum()    # specifically, the time you'll spend on each type of transportation/'mode'
display(mode_mins_0)

journey_0['major_mode'] = mode_mins_0.idxmax()    # the Major Mode is the non-walking mode that takes you the longest time
journey_0

walking legs will be tagged True


0     True
1    False
2    False
3     True
Name: mode_id, dtype: bool

mode_id
bus     14
tube    27
Name: duration, dtype: int64

,startDateTime,duration,legs,fare_totalCost,walking_mins,transport_mins,major_mode
0,2026-10-22T08:04:00,60,[{'$type': 'Tfl.Api.Presentation.Entities.Jour...,565,14,41,tube


dive into the 'journeyVector' column:

not pretty interesting, so just keep the information of 'from' and 'to'. Redundant columns are dropped.

In [24]:
journeyvector_0 = pd.json_normalize(ealing_peak['journeyVector'][0])
type(journeyvector_0)
journeyvector_0 = journeyvector_0.drop(columns=['$type', 'via', 'uri'])
journeyvector_0

,from,to
0,HA01AL,WC2A2AE


Use `pd.concat([], axis=1)` to smash the information from journey, legs, and journeyVector together. Remaining information like the IMD will be added later more elegantly.

I've learned how to extract information for one specific journey, so applying this pattern to all rows will hopefully create a table collecting information for all journeys recorded in a response json file.

In [25]:
ealing_peak_0 = pd.concat([journey_0, journeyvector_0], axis=1)
ealing_peak_0['postcode'] = np.where(ealing_peak_0['from'] != 'WC2A2AE', ealing_peak_0['from'], ealing_peak_0['to'])
ealing_peak_0.drop(columns=['from', 'to'])

,startDateTime,duration,legs,fare_totalCost,walking_mins,transport_mins,major_mode,postcode
0,2026-10-22T08:04:00,60,[{'$type': 'Tfl.Api.Presentation.Entities.Jour...,565,14,41,tube,HA01AL


### 2.1.2, A Function to handle the previous Workflow for all Journeys
'All is one and one is all.' The function that picks information for all borough_timeband.json files collected from NB01.

Then the function is named `pick_info()`, with one only input **borough_timeband** as a pd.DataFrame read from a raw json file.

The principles of each part of the function will be debriefed under the code cell.

In [26]:
def pick_info(borough_timeband):
    # screen the first useable row
    first_valid = None
    first_valid_idx = 0

    for idx in range(len(borough_timeband['journeys'])):    # loops are only here to apply chunks of vectorised operations to all rows
        try:
            candidate = pd.json_normalize(borough_timeband['journeys'][idx], sep='_')
            candidate = candidate[candidate['duration'] == candidate['duration'].min()].head(1)
            if 'fare_totalCost' in candidate.columns:
                first_valid = candidate[['startDateTime', 'duration', 'legs', 'fare_totalCost']]
                first_valid_idx = idx
                break
        except (NotImplementedError, TypeError):
            continue
    
    # the first (useable) row
    journey_0 = first_valid
    journey_0 = journey_0[journey_0['duration'] == journey_0['duration'].min()].head(1)
    journey_0 = journey_0[['startDateTime', 'duration', 'legs', 'fare_totalCost']]

    journey_0_legs = pd.json_normalize(journey_0['legs'].iloc[0], sep='_')
    walking_legs = journey_0_legs[journey_0_legs['mode_id'] == 'walking']
    non_walking_legs = journey_0_legs[journey_0_legs['mode_id'] != 'walking']
    journey_0['walking_mins'] = walking_legs['duration'].sum()
    journey_0['transport_mins'] = non_walking_legs['duration'].sum()
    mode_mins_0 = non_walking_legs.groupby('mode_id')['duration'].sum()
    journey_0['major_mode'] = mode_mins_0.idxmax()

    journeyvector_0 = pd.json_normalize(borough_timeband['journeyVector'][first_valid_idx])
    journeyvector_0 = journeyvector_0.drop(columns=['$type', 'via', 'uri'])
    borough_timeband_clean = pd.concat([journey_0, journeyvector_0], axis=1)
    borough_timeband_clean['postcode'] = np.where(borough_timeband_clean['from'] != 'WC2A2AE', borough_timeband_clean['from'], borough_timeband_clean['to'])

    # following rows
    for i in range(first_valid_idx + 1, len(borough_timeband)):    # a loop is here is simply for managing every row using a set order of vectorised manipulations
        try:
            journey_i = pd.json_normalize(borough_timeband['journeys'][i], sep='_')
        except (NotImplementedError, TypeError) as e:
            print(f"Skipping journey {i}: {e}")
            continue

        journey_i = journey_i[journey_i['duration'] == journey_i['duration'].min()].head(1)

        if 'fare_totalCost' in journey_i.columns:
            journey_i = journey_i[['startDateTime', 'duration', 'legs', 'fare_totalCost']]

            journey_i_legs = pd.json_normalize(journey_i['legs'].iloc[0], sep='_').reset_index(drop=True)
            walking_legs = journey_i_legs[journey_i_legs['mode_id'] == 'walking']
            non_walking_legs = journey_i_legs[journey_i_legs['mode_id'] != 'walking']
            journey_i['walking_mins'] = walking_legs['duration'].sum()
            journey_i['transport_mins'] = non_walking_legs['duration'].sum()
            mode_mins_i = non_walking_legs.groupby('mode_id')['duration'].sum()
            journey_i['major_mode'] = mode_mins_i.idxmax()

            journeyvector_i = pd.json_normalize(borough_timeband['journeyVector'][i])
            journeyvector_i = journeyvector_i.drop(columns=['$type', 'via', 'uri'])
            borough_timeband_i = pd.concat([journey_i, journeyvector_i], axis=1)
            borough_timeband_i['postcode'] = np.where(borough_timeband_i['from'] != 'WC2A2AE', borough_timeband_i['from'], borough_timeband_i['to'])
            borough_timeband_clean = pd.concat([borough_timeband_clean, borough_timeband_i], ignore_index=True)    # attach each processed row to the organised dataframe
        else:
            pass

    # aftermath
    borough_timeband_clean = borough_timeband_clean.dropna()
    borough_timeband_clean = borough_timeband_clean.drop(columns=['from', 'to'])
    borough_timeband_clean['startDateTime'] = pd.to_datetime(borough_timeband_clean['startDateTime'])
    borough_timeband_clean['timeband'] = np.where((borough_timeband_clean['startDateTime'].dt.hour.isin([8, 17])), 'peak', 'offpeak')
    borough_timeband_clean['complexity'] = borough_timeband_clean['legs'].apply(len)
    borough_timeband_clean = borough_timeband_clean.drop(columns='legs')

    return borough_timeband_clean

The pick_info() function's principle:
- the function features 4 parts: `screening, 1st row, following rows, and aftermath`

1. Screening:
   The idea is to screen for the **first useable row**, which can be normalised normally and processed following the workflows demonstrated in the previous section.
   
   This section loops row by row from the first row onwards. It tries to normalise each row and pick the shortest journey as the representation for the route. It also extracts necessary columns if the row has a valide 'fare_totalCost' column. But if a row fails to meet these conditions, the function will simply disregard the row and go on. It will only break and turn to the next step if it finds the first useable row.

2. the 1st Row:
   The idea is to replicate what I did for the experiment/trial row in the last section to all rows from the input dataframe, so the codes are pretty much the same.

   The only difference is that I computed 'walking mins' and 'transport mins' here using direct dictionary structures and .sum(), instead of using the .loc[] mod.

3. the Following Rows:
   A similar screening procedure is added just like before the first row. The function will only proceed with a row if the row can be normalised without any issue and if a 'fare_totalCost' column exists. These operations necessarily avoids any rows with potential structural issues or missing values. If any of these conditions fails, then the function disregards the row and proceeds onwards. The function prints out the journeys skipped in the middle.
   
   Afterwards, the operations are pretty much the same for each row. At the end of this operation, each successful row will be concatenated to the previous, successful dataframe to update the dataframe. This is very simple with pd.concat().

   Also, note that the 'from' and 'to' columns will be replaced by a 'postcode' column that records the other end of the journey in outer London paired with LSE. This is done, agai, using `np.where()`: if the 'from' postcode is not LSE, then use the 'from' postcode; otherwise, use the 'to' postcode. Such differences in 'from' and 'to' are because I collected both inwards and outwards journeys in NB01. `Creating this 'postcode' column is very essential for later averaging.`

4. Aftermath:
   - Check again for missing values: `.dropna()` would drop any rows with empty (NaN) entries.
   - redundant rows like 'from' and 'to' (replaced by 'postcode') will be dropped
   - After converting the startDateTime literally to the datetime format, a 'peak/offpeak' tag will be added using `np.where`. Then the 'startDateTime' column will be dropped
   - the 'complexity' column is added by counting the number of legs/transits for each journey route, and then 'legs' is dropped as a column.

**Reflections**🤔:

Initially, the function was not pretty different from the manual workflow in the last section. But it went through a lot of trials and errors to become the current version. The current function has experienced `3 major upgrades` during the later extension workflow:

1. Adding the **screening process for the 'following' section**. This was because when I experimented the initial function with Enfield and Sutton, some naughty rows in the middle of the dataframes kept bouncing out NotImplemented or 'column not found' errors. To kick off this problem, I chatted with Claude and modified on this screening process.
2. Using the **direct dictionary-structure access rather than the .loc[] method**: the .loc[] method kept bouncing out errors regarding pandas' requirements on Boolean operations. Also by chatting to Claude, I learned that pandas seems to be pretty harsh with Boolean operations and followed the advice to change the way of my operations.
3. Adding the **screening process before the first row**. This was because when I applied the function to Barking, whereas there was no screening for the first row, the first row didn't work. Then I mimiced the screening process for the later rows and created the screening for the first row.

### 2.1.x, an Easter Egg
Display the result of the pick_info dataframe and you'll find for each journey:

`duration >= walking_mins + transport_mins`

Actually this little thing has been here since the start of data collection. It seems that TfL has a special "Allowance" for your journey duration, and the "allowance" was differnet across different journeys. Maybe in NB03 this could be something worth studhying as well.

The two code cells below feature two checking loops. The loops check, whether in any of the rows of ealing_peak_clean and ealing_offpeak_clean the stated inequality (above, duration, walking mins and transport mins) was violated. The answer was that there was no violation.

In [27]:
ealing_peak_clean = pick_info(ealing_peak)
display(ealing_peak_clean.head())
# check for violations
for i in range(len(ealing_peak_clean)):
    if ealing_peak_clean['walking_mins'].iloc[i] + ealing_peak_clean['transport_mins'].iloc[i] > ealing_peak_clean['duration'].iloc[i]:
        print(f'row {i} disobeys the hypothesis')

,startDateTime,duration,fare_totalCost,walking_mins,transport_mins,major_mode,postcode,timeband,complexity
0,2026-10-22 08:04:00,60.0,565.0,14.0,41.0,tube,HA01AL,peak,4
1,2026-10-22 08:02:00,62.0,565.0,16.0,41.0,tube,HA01AX,peak,4
2,2026-10-22 08:02:00,62.0,565.0,16.0,41.0,tube,HA01AY,peak,4
3,2026-10-22 08:02:00,62.0,565.0,21.0,36.0,tube,HA01BS,peak,4
4,2026-10-22 08:02:00,62.0,565.0,21.0,36.0,tube,HA01BT,peak,4


In [28]:
ealing_offpeak = pd.read_json('../data/raw/ealing_offpeak_data.json')
ealing_offpeak_clean = pick_info(ealing_offpeak)
display(ealing_offpeak_clean.head())
# check for violations
for i in range(len(ealing_offpeak_clean)):
    if ealing_offpeak_clean['walking_mins'].iloc[i] + ealing_offpeak_clean['transport_mins'].iloc[i] > ealing_offpeak_clean['duration'].iloc[i]:
        print(f'row {i} disobeys the hypothesis')

,startDateTime,duration,fare_totalCost,walking_mins,transport_mins,major_mode,postcode,timeband,complexity
18,2026-10-22 12:02:00,49.0,485.0,10.0,30.0,tube,NW104XA,offpeak,4
19,2026-10-22 12:01:00,50.0,485.0,11.0,30.0,tube,NW104XB,offpeak,4
20,2026-10-22 12:02:00,52.0,310.0,32.0,20.0,tube,NW106BF,offpeak,3
21,2026-10-22 12:07:00,47.0,485.0,15.0,26.0,tube,NW106DE,offpeak,4
22,2026-10-22 12:02:00,47.0,310.0,25.0,22.0,tube,NW106DJ,offpeak,3


### 2.1.3, Finalising for the Ealing Borough

Concatenate the peak and offpeak tables together:

In [29]:
ealing_clean = pd.concat([ealing_peak_clean, ealing_offpeak_clean], ignore_index=True)
ealing_clean.head()

,startDateTime,duration,fare_totalCost,walking_mins,transport_mins,major_mode,postcode,timeband,complexity
0,2026-10-22 08:04:00,60.0,565.0,14.0,41.0,tube,HA01AL,peak,4
1,2026-10-22 08:02:00,62.0,565.0,16.0,41.0,tube,HA01AX,peak,4
2,2026-10-22 08:02:00,62.0,565.0,16.0,41.0,tube,HA01AY,peak,4
3,2026-10-22 08:02:00,62.0,565.0,21.0,36.0,tube,HA01BS,peak,4
4,2026-10-22 08:02:00,62.0,565.0,21.0,36.0,tube,HA01BT,peak,4


Take averages for inward and outward journeys:

This helps compensate for the slight difference between travelling from A to B and from B to A and reveal the average trend. Note that for 'major_mode', the first entry of the modal value (or values, if there happens to be multiple) of the inward and outward journeys has been taken.

P.S., I don;t know why the .mode()[0] code seems to malfunction here a little bit. But no worries because using the finalised version of the functions (see the Extension workflow) seems to resolve this problem miracally.

In [30]:
ealing_clean = ealing_clean.groupby(['postcode', 'timeband']).agg(
    duration=('duration', 'mean'),
    fare_totalCost=('fare_totalCost', 'mean'),
    complexity=('complexity', 'mean'),
    walking_mins=('walking_mins', 'mean'),
    transport_mins=('transport_mins', 'mean'),
    major_mode=('major_mode', lambda x: x.mode()[0])
).reset_index()
display(ealing_clean.head())
len(ealing_clean)

,postcode,timeband,duration,fare_totalCost,complexity,walking_mins,transport_mins,major_mode
0,HA01AL,offpeak,58.0,505.0,4.0,15.0,35.0,tube
1,HA01AL,peak,60.0,565.0,4.0,14.0,41.0,tube
2,HA01AX,offpeak,59.0,505.0,4.0,16.0,35.0,tube
3,HA01AX,peak,62.0,565.0,4.0,16.0,41.0,tube
4,HA01AY,offpeak,59.0,505.0,4.0,16.0,35.0,tube


50

`Final Cleaning of Data: Dropping Uncomparable Postcodes`:

There are still postcodes that have only valid peak or offpeak records. These postcodes cannot be used for comparisons because there simply exist no matching elements for these if we group the data by peak and offpeak timebands. Comparing these singular postcodes will cause issues relating to unstandardised settings/uncontroled variables and fundamentally tweak the results in a negative way. So these postcodes must be dropped.

Group by all postcodes and Count the number of appearances of each postcode with regards to peak and offpeak timebands. A useable postcode shou;d appear exactly twice with a peak and an offpeak timeband respectively. Drop those that did not appear twice.

In [31]:
postcode_counts = ealing_clean.groupby(['postcode'])['timeband'].count()
useable = postcode_counts[postcode_counts == 2].index
ealing_clean = ealing_clean[ealing_clean['postcode'].isin(useable)]
len(ealing_clean)

32

**Reflections**🤔:

I wanted to do this approach by 'transposing' the table using `.pivot()`. Then journeys with only one peak/offpeak entry will appear in columns with empty entries. I could then drop those columns. However, I found it really annoying transposing the clean matrix back. Then I abandoned this method. The code cell below shows the abandoned method.

In [32]:
# ealing_clean = ealing_clean.pivot(index='timeband', columns='postcode', values=['duration', 'fare_totalCost', 'complexity'])
# ealing_clean
# ealing_clean.dropna(axis=1)

From ons_ealing, retrieve corresponding information of LSOA, usertype, IMD, and borough name.

In [33]:
ons_ealing = pd.read_csv('../data/raw/ons_ealing.csv').drop(columns=['oslaua', 'msoa11', 'lat', 'long'])
ons_ealing.head(3)

,pcds,lsoa11,usertype,imd,borough
0,HA0 1AL,E01001308,1,22868,ealing
1,HA0 1AX,E01001308,0,22868,ealing
2,HA0 1AY,E01001308,0,22868,ealing


Use a `inner merge` to match all valid postcodes with ons information pieces. Unmatching postcodes will be kicked off ('inner').

In [34]:
ons_ealing['pcds'] = ons_ealing['pcds'].str.strip().str.upper().str.replace(' ', '')
ealing = pd.merge(
    ealing_clean,
    ons_ealing,
    left_on='postcode',
    right_on='pcds',
    how='inner'
).drop(columns=['pcds', 'lsoa11'])

ealing

,postcode,timeband,duration,fare_totalCost,complexity,walking_mins,transport_mins,major_mode,usertype,imd,borough
0,HA01AL,offpeak,58.0,505.0,4.0,15.0,35.0,tube,1,22868,ealing
1,HA01AL,peak,60.0,565.0,4.0,14.0,41.0,tube,1,22868,ealing
2,HA01AX,offpeak,59.0,505.0,4.0,16.0,35.0,tube,0,22868,ealing
3,HA01AX,peak,62.0,565.0,4.0,16.0,41.0,tube,0,22868,ealing
4,HA01AY,offpeak,59.0,505.0,4.0,16.0,35.0,tube,0,22868,ealing
5,HA01AY,peak,62.0,565.0,4.0,16.0,41.0,tube,0,22868,ealing
6,HA01BS,offpeak,59.0,535.0,4.0,18.0,36.0,tube,0,19108,ealing
7,HA01BS,peak,61.0,610.0,4.0,19.5,36.5,tube,0,19108,ealing
8,HA01BU,offpeak,58.0,535.0,4.0,17.0,36.0,tube,0,19108,ealing
9,HA01BU,peak,59.0,655.0,4.0,17.0,37.0,tube,0,19108,ealing


## 2.2, EXTENSION: applying tidy-up functions to other Outer London Boroughs
Results are saved as a .csv file in the data/processed directory. Relative path from the notebooks to the file: "../data/processed/processed_data.csv"

### 2.2.1, Another Function to standardise the workflow for all boroughs
This function uses the `pick_info()` function directly. The `pick_info()` function have been defined in section 2.1.

This function reads the raw json files for each borough's peak and offpeak data, and process those data with the `pick_info()` function. After that, it takes **averages for inward and outward journeys** and takes the **major mode** as the first entry of the modal value (or values if there happen to be multiples).

Then the function counts the number of appearances of each postcode to check the validity of data: only if a postcode has occurred twice in the dataframe (i.e., for both 'peak' and 'offpeak' timebands) will they be preserved.

Finally the function merges journeys' information with their corresponding ONS descriptions and drops unnecessary columns.

**This function is called `tidy_up()`, with only one input "*borough*" as a string.**

In [35]:
def tidy_up(borough):
    borough_peak = pd.read_json(f'../data/raw/{borough}_peak_data.json')
    borough_offpeak = pd.read_json(f'../data/raw/{borough}_offpeak_data.json')
    ons_borough = pd.read_csv(f'../data/raw/ons_{borough}.csv').drop(columns=['oslaua', 'msoa11', 'lat', 'long'])

    borough_peak_clean = pick_info(borough_peak)
    borough_offpeak_clean = pick_info(borough_offpeak)

    borough_clean = pd.concat([borough_peak_clean, borough_offpeak_clean], ignore_index=True)
    borough_clean = borough_clean.groupby(['postcode', 'timeband']).agg(
        duration=('duration', 'mean'), 
        fare_totalCost=('fare_totalCost', 'mean'), 
        complexity=('complexity', 'mean'),
        walking_mins=('walking_mins', 'mean'),
        transport_mins=('transport_mins', 'mean'),
        major_mode=('major_mode', lambda x: x.mode()[0])
    ).reset_index()

    postcode_counts = borough_clean.groupby(['postcode'])['timeband'].count()
    useable = postcode_counts[postcode_counts == 2].index
    borough_clean = borough_clean[borough_clean['postcode'].isin(useable)]

    ons_borough['pcds'] = ons_borough['pcds'].str.strip().str.upper().str.replace(' ', '')
    borough = pd.merge(
    borough_clean,
    ons_borough,
    left_on='postcode',
    right_on='pcds',
    how='inner'
    )

    borough = borough.drop(columns=['pcds', 'lsoa11'])

    return borough

**Reflections**🤔:
Initially when processing the 'major mode' for the averaged dataframe, I found it quite confusing: sometimes you really can't avoid the inward and outward journeys having different major modes (though basically the major modes are just different subcategories of the rail system: underground/tude, elizabeth line, national rail, London overground). Then I adopted Claude's suggestion of using the first entry of the modal values of the major modes to resolve this problem.

### 2.2.2, Ealing

In [36]:
ealing = tidy_up('ealing')
ealing

,postcode,timeband,duration,fare_totalCost,complexity,walking_mins,transport_mins,major_mode,usertype,imd,borough
0,HA01AL,offpeak,58.0,505.0,4.0,15.0,35.0,tube,1,22868,ealing
1,HA01AL,peak,60.0,565.0,4.0,14.0,41.0,tube,1,22868,ealing
2,HA01AX,offpeak,59.0,505.0,4.0,16.0,35.0,tube,0,22868,ealing
3,HA01AX,peak,62.0,565.0,4.0,16.0,41.0,tube,0,22868,ealing
4,HA01AY,offpeak,59.0,505.0,4.0,16.0,35.0,tube,0,22868,ealing
5,HA01AY,peak,62.0,565.0,4.0,16.0,41.0,tube,0,22868,ealing
6,HA01BS,offpeak,59.0,535.0,4.0,18.0,36.0,tube,0,19108,ealing
7,HA01BS,peak,61.0,610.0,4.0,19.5,36.5,tube,0,19108,ealing
8,HA01BU,offpeak,58.0,535.0,4.0,17.0,36.0,tube,0,19108,ealing
9,HA01BU,peak,59.0,655.0,4.0,17.0,37.0,tube,0,19108,ealing


**Reflections**🤔:
I carefully browsed through this table to check whether there were any ridiculous values:
- duration should be around or under one hour
- fare total costs are similar
- walking mins + transport mins <= duration (all averaged)
- major modes are all single entries, no list entry
- a mix of both usertypes (0 and 1)
- IMD in range: 1 ~ 30000-ish

This process is extremely important as last time for mini-project one, I did not browse for any ridiculous values. Then I made mistakes while analysing average trends because I included -9999 entries for data, which did not make any sense but was returned by that stupid API. I should not be stupid again here.

### 2.2.3, Enfield
note that the skipped journeys are printed out

In [42]:
enfield = tidy_up('enfield')
display(enfield)

Skipping journey 12: 
Skipping journey 15: 
Skipping journey 37: 
Skipping journey 40: 
Skipping journey 12: 
Skipping journey 15: 
Skipping journey 37: 
Skipping journey 40: 


,postcode,timeband,duration,fare_totalCost,complexity,walking_mins,transport_mins,major_mode,usertype,imd,borough
0,EN11AA,offpeak,57.0,467.5,5.5,14.5,31.0,tube,1,15781,enfield
1,EN11AA,peak,58.0,455.0,5.0,20.5,29.0,overground,1,15781,enfield
2,EN11BA,offpeak,51.5,467.5,5.5,13.0,27.0,tube,0,11086,enfield
3,EN11BA,peak,52.0,380.0,5.0,18.0,25.0,tube,0,11086,enfield
4,EN11BB,offpeak,52.5,467.5,5.5,14.0,27.0,tube,0,11086,enfield
5,EN11BB,peak,53.0,380.0,5.0,19.0,25.0,tube,0,11086,enfield
6,EN11BE,offpeak,55.5,467.5,5.5,17.0,27.0,tube,0,12968,enfield
7,EN11BE,peak,56.0,380.0,5.0,22.0,25.0,tube,0,12968,enfield
8,EN11BG,offpeak,55.5,467.5,5.5,17.0,27.0,tube,0,11086,enfield
9,EN11BG,peak,56.0,380.0,5.0,22.0,25.0,tube,0,11086,enfield


### 2.2.4, Sutton

In [43]:
sutton = tidy_up('sutton')
sutton

Skipping journey 20: 
Skipping journey 41: 
Skipping journey 20: 
Skipping journey 41: 


,postcode,timeband,duration,fare_totalCost,complexity,walking_mins,transport_mins,major_mode,usertype,imd,borough
0,CR03AQ,offpeak,67.0,545.0,6.0,17.0,43.0,national-rail,0,15853,sutton
1,CR03AQ,peak,66.0,835.0,8.0,20.0,43.0,national-rail,0,15853,sutton
2,CR03AR,offpeak,64.0,545.0,6.0,14.0,43.0,national-rail,0,15853,sutton
3,CR03AR,peak,68.0,835.0,8.0,20.0,45.0,bus,0,15853,sutton
4,CR03AS,offpeak,69.0,525.0,7.0,20.5,43.5,bus,0,15853,sutton
5,CR03AS,peak,69.0,835.0,8.0,21.0,45.0,bus,0,15853,sutton
6,CR04AL,offpeak,67.0,615.0,8.0,17.0,45.0,bus,0,20472,sutton
7,CR04AL,peak,67.0,835.0,8.0,17.0,45.0,bus,0,20472,sutton
8,CR04AQ,offpeak,66.0,615.0,8.0,18.0,43.0,national-rail,0,20472,sutton
9,CR04AQ,peak,66.0,835.0,8.0,18.0,43.0,national-rail,0,20472,sutton


### 2.2.5, Barking

In [44]:
barking = tidy_up('barking')
barking

Skipping journey 12: 
Skipping journey 32: 
Skipping journey 12: 
Skipping journey 32: 


,postcode,timeband,duration,fare_totalCost,complexity,walking_mins,transport_mins,major_mode,usertype,imd,borough
0,IG110AE,offpeak,66.0,535.0,5.0,26.0,34.0,tube,0,3997,barking
1,IG110AE,peak,66.0,535.0,5.0,26.0,34.0,tube,0,3997,barking
2,IG110AG,offpeak,62.0,535.0,5.0,22.0,34.0,tube,0,2669,barking
3,IG110AG,peak,63.0,507.5,4.5,28.0,31.5,tube,0,2669,barking
4,IG110AN,offpeak,62.0,535.0,5.0,22.0,34.0,tube,0,2669,barking
5,IG110AN,peak,63.0,480.0,4.0,33.0,29.0,tube,0,2669,barking
6,IG110AQ,offpeak,62.0,535.0,5.0,22.0,34.0,tube,0,2669,barking
7,IG110AQ,peak,63.0,507.5,4.5,28.0,31.5,tube,0,2669,barking
8,IG117BS,offpeak,53.0,535.0,4.0,12.0,35.0,tube,1,5880,barking
9,IG117BS,peak,56.0,595.0,5.0,18.0,31.5,tube,1,5880,barking


### 2.2.6, Save
Data for all boroughs are put together and are saved in the data/processed directory.

In [45]:
processed = pd.concat([ealing, enfield])
processed = pd.concat([processed, sutton])
processed = pd.concat([processed, barking])
display(processed)
print(len(ealing) + len(enfield) + len(sutton) + len(barking))

,postcode,timeband,duration,fare_totalCost,complexity,walking_mins,transport_mins,major_mode,usertype,imd,borough
0,HA01AL,offpeak,58.0,505.0,4.0,15.0,35.0,tube,1,22868,ealing
1,HA01AL,peak,60.0,565.0,4.0,14.0,41.0,tube,1,22868,ealing
2,HA01AX,offpeak,59.0,505.0,4.0,16.0,35.0,tube,0,22868,ealing
3,HA01AX,peak,62.0,565.0,4.0,16.0,41.0,tube,0,22868,ealing
4,HA01AY,offpeak,59.0,505.0,4.0,16.0,35.0,tube,0,22868,ealing
...,...,...,...,...,...,...,...,...,...,...,...
15,IG117ND,peak,48.5,420.0,4.0,20.5,26.5,tube,1,6900,barking
16,IG117PB,offpeak,52.0,360.0,4.0,24.0,27.0,tube,1,5228,barking
17,IG117PB,peak,52.0,360.0,4.0,24.0,27.0,tube,1,5228,barking
18,IG117RS,offpeak,56.0,535.0,5.0,15.0,35.0,tube,1,3117,barking


112


The lengths match. Save as a csv file.

In [46]:
processed.to_csv('../data/processed/processed_data.csv', index=False)